In [ ]:
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import cartopy
import pandas as pd
from shapely.ops import unary_union
import cartopy.feature as cfeature
import geopandas as gpd
from shapely.geometry import mapping
from shapely.geometry import Polygon
import rioxarray
import sys
sys.path.append(r'C:\Users\grace.davis\Documents\GitHub\RESOURCES\python')
import utilities
from utilities import get_prod_files
import cmocean
from matplotlib.colors import LogNorm
import cartopy.crs as crs
import statsmodels as sm
from statsmodels import nonparametric
from statsmodels.nonparametric import smoothers_lowess
import scipy
from scipy import signal
from scipy.signal import find_peaks
from scipy.signal import argrelextrema

### Calculating the Threshold Value

In [ ]:
#Define a function to return the threshold value based on the regional climatology.
def threshold_value(thld = 0.05, path = None):
    if path == None:
        file = get_prod_files('CHL',mapping='NES',period='ANNUAL') #finds path for the annual climatology file
        base_ds = xr.open_dataset(file[0]) #opens netCDF file
    else:
        base_ds = xr.open_dataset(path)
    median_CHL = base_ds.CHL_median #grabs CHL_median variable
    thld_value = median_CHL*(1+thld) #Sets threshold value per latitude and longitude point
    return(thld_value)

### Creating Bloom Masks

In [ ]:
def bloom_mask_Boolean(path=None): #Produces True and False values
    if path == None:
        files = get_prod_files('CHL',mapping='NES',period='D8') #Finds all 8 day rolling mean files for NWA
        ds = xr.open_mfdataset(files)
    else:
        ds = xr.open_mfdataset(path)
    med_CHL = ds.CHL_median #Extracts CHL_mean variable for the files
    is_bloom_CHL = med_CHL > clim_med #Is the median chl-a in each files greater than the climatological mean? Creates a Boolean array of trues and falses.
    return is_bloom_CHL

In [ ]:
def bloom_mask_numeric(path=None): #Produces 0 and actual values
    if path == None:
        files = get_prod_files('CHL',mapping='NES',period='D8') #Finds all 8 day rolling mean files for NWA
        ds = xr.open_mfdataset(files)
    else:
        #ds = xr.open_mfdataset(path)
        ds = xr.open_zarr(path,consolidated=True) #Opens zarr file, use until mfdatasets is working properly
    med_CHL = ds.CHL_median #Extracts CHL_mean variable for the files
    clim_med_new = clim_med.squeeze('time', drop=True) #Removes time dimension from climatological mean, FIX THIS LINE
    is_bloom_CHL = med_CHL.where(med_CHL > clim_med_new, 0) #Turns false into 0 and trues retain their value
    return is_bloom_CHL

### Creating subsets of data for histogram plotting

In [ ]:
def hist_local_chl(lat_min,lat_max,lon_min,lon_max,path=None,region_title=None):
    if path == None:
        #daily_data = xr.open_mfdataset(r'C:\Users\grace.davis\Documents\GitHub\DATASETS\OCCCI\V6.0\PRODUCTS\NES_4KM_DAY8\CHL\D8*.nc')
        daily_data = xr.open_zarr(r'C:\Users\grace.davis\Documents\GitHub\DATASETS\OCCCI\V6.0\PRODUCTS\NES_4KM_DAY8\CHL\D8_COMBINED.zarr',consolidated=True)
    else:
        daily_data = xr.open_mfdataset(path)
    lat_min = lat_min
    lat_max = lat_max
    lon_min = lon_min
    lon_max = lon_max
    daily_data_local = daily_data.CHL_median.sel(
    lat=slice(lat_min,lat_max),
    lon=slice(lon_min,lon_max)
    )
    daily_data_local = daily_data_local.mean(dim=['lat','lon'])
    return daily_data_local

In [ ]:
def hist_clim_local(lat_min,lat_max,lon_min,lon_max,region_title=None,path=None):
    if path == None:
        clim = xr.open_dataset(r'C:\Users\grace.davis\Documents\GitHub\DATASETS\OCCCI\V6.0\PRODUCTS\NES_4KM_CLIMATOLOGY\CHL\ANNUAL_1998_2020-OCCCI-CHL-NES-STATS.nc')
    else:
        clim = xr.open_dataset(path)
    clim_med = clim.CHL_median
    lat_min = lat_min
    lat_max = lat_max
    lon_min = lon_min
    lon_max = lon_max
    clim_med = clim_med.sel(
    lat=slice(lat_min,lat_max),
    lon=slice(lon_min,lon_max)
    )
    clim_med_bounded = clim_med.mean(dim=['lat','lon'])
    return clim_med_bounded

In [ ]:
def bound_local(lat_min,lat_max,lon_min,lon_max): #Builds polygon shape for the map
    coords = [(lon_min,lat_min),
              (lon_max,lat_min),
              (lon_max,lat_max),
              (lon_min,lat_max),
              (lon_min,lat_min)]
    box_polygon = Polygon(coords)
    return box_polygon

### Determining the percentage of data points above a threshold value

In [ ]:
def percent_above_thld(threshold,index,data):
    threshold=threshold[index].values
    total_above = int((data>threshold).sum())
    percent = (total_above/len(data))*100
    return percent

### Finding the percent deviation from the median

In [ ]:
def percent_deviation(dataset,index,median):
    top = dataset.squeeze()-median[index][0]
    fraction = top/median[index][0]
    percent_dev = fraction*100
    return percent_dev

Making an addition to a netCDF file

In [ ]:
#Adding variable to netCDF file
#from netCDF4 import Dataset
#path = r'C:\Users\grace.davis\Documents\GitHub\DATASETS\OCCCI\V6.0\PRODUCTS\NWA_4KM_DAY8\CHL\D8_19980102_19980109-OCCCI-CHL-NWA-STATS.nc'
#ds = xr.open_dataset(path)
#cdf = bloom_mask_numeric(path=path) #Creates masked variable
#ds['five_percent_thld'] = cdf #Adds variable to ds
#ds.to_netcdf(r'C:\Users\grace.davis\Documents\GitHub\DATASETS\OCCCI\V6.0\PRODUCTS\NWA_4KM_DAY8\CHL\D8_19980102_19980109-OCCCI-CHL-NWA-STATS_with_threshold.nc') #Renames the file and saves it with the new variable

### Smoothing the data

In [ ]:
def smoothing_data(dataset):
    time = dataset.time.astype('int64') #Changes time values to integers for smoothing
    median=dataset.to_dataframe(name='Chl_a') #Converts it into a pandas dataframe
    smoothed_median = sm.nonparametric.smoothers_lowess.lowess(median['Chl_a'],time,frac=0.04)
    return smoothed_median

### Finding rate of change for specific lat and lon coordinates

In [ ]:
def rate_of_change(lat_min,lat_max,lon_min,lon_max,path=None):
    if path == None:
        files = get_prod_files('CHL',map_region='NES',period='WEEK')
        clim_med = xr.open_mfdataset(files)
    else:
        clim_med = xr.open_mfdataset(path)
    clim_med = clim_med.CHL_median
    clim_med = hist_local_chl(lat_min,lat_max,lon_min,lon_max,path)
    time = clim_med.time.astype('int64') #Changes time values to integers for smoothing
    clim_median=clim_med.to_dataframe(name='Chl_a') #Converts it into a pandas dataframe
    smoothed_median = sm.nonparametric.smoothers_lowess.lowess(clim_median['Chl_a'],time,frac=0.03)
    smoothed_CHL = smoothed_median[:,1] #Extracts Chl-a data from smooth curve
    clim_roc = np.gradient(smoothed_CHL) #Calculates rate of change
    return clim_roc

In [ ]:
def max_roc(lat_min,lat_max,lon_min,lon_max,path,days=10,prm=0.02): #Days between peaks and the relative height of each peak value
    roc = rate_of_change(lat_min,lat_max,lon_min,lon_max,path)
    roc_max = find_peaks(roc,distance=days,prominence=prm)
    return roc_max

### Regional rate of change

In [ ]:
def instant_rate_of_change(dataset,path=None):
    time = dataset.time.astype('int64') #Changes time values to integers for smoothing
    median=dataset.to_dataframe(name='Chl_a') #Converts it into a pandas dataframe
    smoothed_median = sm.nonparametric.smoothers_lowess.lowess(median['Chl_a'],time,frac=0.04)
    smoothed_CHL = smoothed_median[:,1] #Extracts Chl-a data from smooth curve
    roc = np.gradient(smoothed_CHL) #Calculates rate of change
    return roc

In [ ]:
def max_roc_dates(dataset,prm=0.015,days=10,): #Days between peaks and the relative height of each peak value
    roc = instant_rate_of_change(dataset)
    roc_max = find_peaks(roc,distance=days,prominence=prm)
    return roc_max

In [ ]:
def max_roc_for_bloom(dataset,index,clim=clim_10,prm=0.02,days=10):
    start_day_bloom=initiation_date(dataset,index)[0]
    end_day_bloom=initiation_date(dataset,index)[1]
    roc = instant_rate_of_change(dataset)
    bloom_events = []
    for i in range(len(start_day_bloom)):
        start_day=start_day_bloom[i]
        if i <len(end_day_bloom):
            end_day=end_day_bloom[i]
        else:
            continue
        range_roc = roc[start_day:end_day]
        if len(range_roc)>0:
            range_max_roc = np.argmax(range_roc) #Finds local maximum rate of change for each detected bloom
            max_roc_index = start_day+range_max_roc #Gets the actual day of year value
            bloom_events.append(max_roc_index)
    return bloom_events

### Peak Detection

In [ ]:
def bloom_peak_detection(dataset,index,days=10,prm=0.02,):
    smoothed_median = smoothing_data(dataset)
    smoothed_CHL = smoothed_median[:,1] #Extracts Chl-a data from smooth curve
    chl_peak_loc, _ = find_peaks(smoothed_CHL,distance=days,prominence=prm) #Finds all peaks
    clim_threshold = clim_10[index].values
    chl_peaks = []
    for i in range(0,len(chl_peak_loc)): #Finds all peaks above the threshold and removes peaks below the threshold
        if smoothed_CHL[chl_peak_loc[i]]>clim_threshold[0]:
            chl_peak_location = chl_peak_loc[i]
            chl_peaks.append(chl_peak_location)
        else:
            continue
    return chl_peaks

In [ ]:
def bloom_event_detection(dataset,index,event_distance=14):
    chl_peaks = bloom_peak_detection(dataset,index)
    bloom_events = []
    current_event = [chl_peaks[0]]
    days_between_events = event_distance
    for i in range(1,len(chl_peaks)):
        if chl_peaks[i]-chl_peaks[i-1]<days_between_events: #Checks the distance between peaks
            current_event.append(chl_peaks[i]) #Groups close peaks into one event
        else:
            bloom_events.append(current_event)
            current_event = [chl_peaks[i]] #If peak is not close to other peaks, it adds it to the event list by itself
    bloom_events.append(current_event)
    return bloom_events

### Bloom Initiation and Termination Dates

In [ ]:
def initiation_date(dataset,index,days=10,prm=0.02,peak_window=10,event_distance=14):
    data = dataset
    smoothed_data = smoothing_data(data)
    smoothed_CHL = smoothed_data[:,1]
    roc = instant_rate_of_change(data)
    bloom_events = bloom_event_detection(dataset,index)
    start_DOY = []
    end_DOY = []
    window_for_peak = peak_window
    last_end_day=0
    last_start_day=0
    clim_threshold = clim_10[index]

    for event in bloom_events:
        start_day = 0
        for i in range(event[0]-1,window_for_peak-1,-1):
            if all(roc[i+w]<0 for w in range(window_for_peak)):
                if i>=last_end_day and last_end_day>=last_start_day:
                    start_day = i
                    start_DOY.append(start_day)
                    last_start_day = i
                    break
        if start_day == 0 and last_end_day>0 and event[0]>last_end_day:
                start_day = last_end_day
                start_DOY.append(start_day)
                last_start_day = start_day
        if start_day>0:
            end_day = len(roc)-1
            for i in range(event[-1]+1,len(roc)-window_for_peak,1):
                roc_condition = all(roc[i+w]>=0 for w in range(window_for_peak)) # Ensures that the roc is becoming positive
                threshold_condition = all(smoothed_CHL[i+w]<clim_threshold for w in range(window_for_peak)) # Ensures termination date only occurs after a drop below the climatological median
                if roc_condition and threshold_condition:
                    end_day=i
                    end_DOY.append(end_day)
                    last_end_day=end_day
                    break
    return start_DOY,end_DOY,bloom_events